# 03 - Inference
Loads the packaged pipeline exactly as the API does, predicts for sample patients, then
(optionally) calls the running API. Start the API first if you want the last section:
`uvicorn api.main:app --port 8000` (or the Docker container).

In [ ]:
# Make the repo importable whether Jupyter was started in the repo root or in notebooks/
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "cardiorisk").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

In [ ]:
import json
import joblib
import pandas as pd
from cardiorisk import config as C
from cardiorisk.data import load_clean

model = joblib.load(C.MODEL_PATH)
meta = json.loads(C.METADATA_PATH.read_text())
print("Loaded:", meta["model_name"], meta["model_version"])
model

## 1. Predict on a few real rows (raw features in, probability out - no manual preprocessing)

In [ ]:
df = load_clean()
sample = df.sample(5, random_state=1)
X = sample[C.FEATURES].astype(float)
proba = model.predict_proba(X)[:, list(model.classes_).index(1)]
out = sample[C.FEATURES + [C.TARGET]].copy()
out["p_disease"] = proba.round(3)
out["predicted"] = (proba >= meta["decision_threshold"]).astype(int)
out

## 2. Missing values are handled by the pipeline's imputers

In [ ]:
row = X.iloc[[0]].copy()
row[["chol", "thal"]] = float("nan")
model.predict_proba(row)

## 3. Call the running API (skipped gracefully if it is not running)

In [ ]:
import urllib.request, urllib.error
payload = json.loads((C.ROOT / "scripts" / "sample_request.json").read_text())
try:
    req = urllib.request.Request("http://localhost:8000/predict", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        print(json.dumps(json.loads(r.read()), indent=2))
except (urllib.error.URLError, ConnectionError) as exc:
    print("API not reachable on localhost:8000 - start it first. Details:", exc)